# ***Environment and configuration***

In [1]:
import os

os.environ.setdefault("HF_HOME", "/media/caotulab/303A225B3A221DFA/hf_cache")
os.environ.setdefault("WANDB_PROJECT", "alqac-vnlegal-lal")

import random

import numpy as np
import torch
import wandb

wandb.login()

MODEL_NAME = "darklethelong/vnlegal-lal"
OUTPUT_DIR = "/media/caotulab/303A225B3A221DFA/vnlegal_lal_alqac"
MERGED_DIR = "/media/caotulab/303A225B3A221DFA/vnlegal_lal_alqac_merged"
QUERY_PROMPT = "Instruct: Given a Vietnamese legal question, retrieve relevant legal passages that answer the question\nQuery: "
MAX_SEQ_LENGTH = 2048
K_VALUES = [1, 3, 5, 10]
ANCE_ROUNDS = 3
NEGATIVES_PER_QUERY = 8
DENSE_POOL = 100
BM25_POOL = 100
FALSE_NEGATIVE_MARGIN = 0.05
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

random.seed(42)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/caotulab/.netrc.
wandb: Currently logged in as: syun_1208 (syun12) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


# ***Data preparation***

In [2]:
import json
import re

CORPUS_PATH = os.environ.get("ALQAC_CORPUS", "../data/corpus_law_pub.json")
TEST_PATH = os.environ.get("ALQAC_TEST", "../data/ALQAC2026_public_test.json")
EVAL_HELDOUT_CASES = 10
ICT_SENTENCES_PER_ARTICLE = 1
MIN_SENTENCE_LEN = 25

with open(CORPUS_PATH, "r", encoding="utf-8") as handle:
    documents = json.load(handle)

corpus = {}
number_to_aid = {}
for law in documents:
    law_id = str(law.get("law_id", "")).strip()
    article_no = 0
    for article in law.get("content", []):
        article_no += 1
        number_to_aid[(law_id, article_no)] = article["aid"]
        text = str(article.get("content_Article", "")).strip()
        if text:
            corpus[f"{law_id}::{article['aid']}"] = text

CORPUS_LAW_IDS = {cid.split("::")[0] for cid in corpus}

_CODE_RE = re.compile(r"\d+/\d{4}/[A-Za-zĐđ\-]+")
_ART_RE = re.compile(r"[Đđ]iều\s+(\d+)")
_OUTDATED = ("1987", "1993", "1995", "1998", "2000", "2003", "2004", "2005", "2006", "2009")


def _resolve_law_id(name):
    for match in _CODE_RE.finditer(name):
        if match.group(0) in CORPUS_LAW_IDS:
            return match.group(0)
    low = name.lower()
    outdated = any(year in low for year in _OUTDATED)
    if "tố tụng dân sự" in low:
        return "92/2015/QH13"
    if "tố tụng hành chính" in low:
        return "93/2015/QH13"
    if "dân sự" in low:
        return None if outdated else "91/2015/QH13"
    if "hình sự" in low:
        return "100/2015/QH13"
    if "đất đai" in low:
        return None if outdated else "45/2013/QH13"
    if "hôn nhân" in low:
        return None if outdated else "52/2014/QH13"
    if "án phí" in low or "lệ phí" in low:
        return None if "pháp lệnh" in low else "326/2016/UBTVQH14"
    if "thi hành án" in low:
        return "26/2008/QH12"
    if "hộ tịch" in low:
        return "60/2014/QH13"
    if "khiếu nại" in low:
        return None if ("tố cáo" in low or outdated) else "02/2011/QH13"
    if "tổ chức tín dụng" in low:
        return "47/2010/QH12"
    if "kinh doanh bất động sản" in low:
        return None if outdated else "66/2014/QH13"
    if "xây dựng" in low:
        return "50/2014/QH13"
    return None


def _cited_cids(related_text):
    cids = set()
    for line in str(related_text or "").splitlines():
        if "|" not in line:
            continue
        name, remainder = line.split("|", 1)
        law_id = _resolve_law_id(name.strip())
        if law_id is None:
            continue
        for raw_number in _ART_RE.findall(remainder):
            aid = number_to_aid.get((law_id, int(raw_number)))
            cid = f"{law_id}::{aid}" if aid is not None else None
            if cid in corpus:
                cids.add(cid)
    return cids


labelled_cases = []
if os.path.exists(TEST_PATH):
    with open(TEST_PATH, "r", encoding="utf-8") as handle:
        cases = json.load(handle)
    for case in cases:
        query = str(case.get("case_query", "")).strip()
        cids = _cited_cids(case.get("related_law_provisions", ""))
        if query and cids:
            labelled_cases.append((str(case.get("case_id", "")), query, cids))

random.shuffle(labelled_cases)
eval_cases = labelled_cases[:EVAL_HELDOUT_CASES] if EVAL_HELDOUT_CASES > 0 else []
train_cases = labelled_cases[EVAL_HELDOUT_CASES:] if EVAL_HELDOUT_CASES > 0 else labelled_cases

train_examples = [(query, cids) for (_id, query, cids) in train_cases]

queries = {cid_key: query for (cid_key, query, _cids) in eval_cases}
relevant_docs = {cid_key: set(cids) for (cid_key, _q, cids) in eval_cases}

print(f"corpus: {len(corpus)} articles | labelled cases: {len(labelled_cases)}")
print(f"train cases: {len(train_cases)} | eval cases: {len(eval_cases)}")

corpus: 3352 articles | labelled cases: 50
train cases: 40 | eval cases: 10


# ***Load base model and evaluator***

In [3]:
from sentence_transformers import SentenceTransformer
from sentence_transformers.evaluation import InformationRetrievalEvaluator

model = SentenceTransformer(
    MODEL_NAME,
    device=DEVICE,
    model_kwargs={"torch_dtype": torch.bfloat16},
)
model.max_seq_length = MAX_SEQ_LENGTH

ir_evaluator = InformationRetrievalEvaluator(
    queries=queries,
    corpus=corpus,
    relevant_docs=relevant_docs,
    name="alqac",
    query_prompt=QUERY_PROMPT,
    accuracy_at_k=K_VALUES,
    precision_recall_at_k=K_VALUES,
    ndcg_at_k=[10],
    mrr_at_k=[10],
    map_at_k=[10],
    show_progress_bar=True,
)

/media/caotulab/303A225B3A221DFA/envs/nina/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/media/caotulab/303A225B3A221DFA/Sang_IIPP/tmp/ipykernel_2612455/3091200899.py:2: DeprecationWarning: Importing from 'sentence_transformers.evaluation' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.evaluation' instead.
  from sentence_transformers.evaluation import InformationRetrievalEvaluator
Loading weights: 100%|██████████| 310/310 [00:00<00:00, 1964.78it/s]


In [4]:
base_metrics = ir_evaluator(model)
for key in sorted(base_metrics):
    print(key, base_metrics[key])

Corpus Chunks: 100%|██████████| 1/1 [00:17<00:00, 17.37s/it]

alqac_cosine_accuracy@1 0.0
alqac_cosine_accuracy@10 0.1
alqac_cosine_accuracy@3 0.0
alqac_cosine_accuracy@5 0.0
alqac_cosine_map@10 0.001111111111111111
alqac_cosine_mrr@10 0.01111111111111111
alqac_cosine_ndcg@10 0.006625422345438903
alqac_cosine_precision@1 0.0
alqac_cosine_precision@10 0.01
alqac_cosine_precision@3 0.0
alqac_cosine_precision@5 0.0
alqac_cosine_recall@1 0.0
alqac_cosine_recall@10 0.005555555555555555
alqac_cosine_recall@3 0.0
alqac_cosine_recall@5 0.0


# ***Stage 1: BM25 index construction***

In [5]:
from rank_bm25 import BM25Okapi

corpus_ids = list(corpus.keys())
corpus_texts = [corpus[cid] for cid in corpus_ids]
cid_to_index = {cid: index for index, cid in enumerate(corpus_ids)}


def tokenize(text):
    return text.lower().split()


bm25 = BM25Okapi([tokenize(text) for text in corpus_texts])
print(f"BM25 index built over {len(corpus_texts)} documents")

BM25 index built over 3352 documents


# ***Stage 2: Iterative ANCE training (LoRA + InfoNCE)***

In [6]:
from datasets import Dataset
from peft import LoraConfig, TaskType, get_peft_model

lora_config = LoraConfig(
    task_type=TaskType.FEATURE_EXTRACTION,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model[0].model = get_peft_model(model[0].model, lora_config)
model[0].model.print_trainable_parameters()


def encode_corpus(current_model):
    return current_model.encode(
        corpus_texts,
        batch_size=16,
        convert_to_tensor=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    )


def mine_triplets(current_model, document_embeddings):
    query_texts = [query for query, _cids in train_examples]
    query_embeddings = current_model.encode(
        query_texts,
        prompt=QUERY_PROMPT,
        batch_size=16,
        convert_to_tensor=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    )
    dense_scores = query_embeddings @ document_embeddings.T
    anchors, positives, negatives = [], [], []
    for row, (query, relevant_cids) in enumerate(train_examples):
        relevant_indices = {cid_to_index[cid] for cid in relevant_cids}
        best_positive_score = max(dense_scores[row, index].item() for index in relevant_indices)
        dense_candidates = torch.topk(dense_scores[row], DENSE_POOL).indices.tolist()
        bm25_candidates = bm25.get_top_n(tokenize(query), corpus_ids, n=BM25_POOL)
        bm25_indices = [cid_to_index[cid] for cid in bm25_candidates]
        pool, seen = [], set()
        for index in dense_candidates + bm25_indices:
            if index in relevant_indices or index in seen:
                continue
            if dense_scores[row, index].item() >= best_positive_score - FALSE_NEGATIVE_MARGIN:
                continue
            seen.add(index)
            pool.append(index)
        pool.sort(key=lambda index: dense_scores[row, index].item(), reverse=True)
        chosen = pool[:NEGATIVES_PER_QUERY]
        if not chosen:
            chosen = [index for index in dense_candidates if index not in relevant_indices][:NEGATIVES_PER_QUERY]
        for positive_cid in relevant_cids:
            positive_index = cid_to_index[positive_cid]
            for negative_index in chosen:
                anchors.append(query)
                positives.append(corpus_texts[positive_index])
                negatives.append(corpus_texts[negative_index])
    return Dataset.from_dict({"anchor": anchors, "positive": positives, "negative": negatives})

trainable params: 10,092,544 || all params: 606,142,464 || trainable%: 1.6650


In [7]:
from sentence_transformers import SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from sentence_transformers.losses import CachedMultipleNegativesRankingLoss
from sentence_transformers.training_args import BatchSamplers

train_loss = CachedMultipleNegativesRankingLoss(model, mini_batch_size=4)


def train_round(round_index, triplet_dataset):
    args = SentenceTransformerTrainingArguments(
        output_dir=f"{OUTPUT_DIR}/round_{round_index}",
        num_train_epochs=100,
        per_device_train_batch_size=128,
        gradient_checkpointing=True,
        warmup_ratio=0.1,
        learning_rate=1e-4,
        lr_scheduler_type="cosine",
        optim="adamw_torch_fused",
        bf16=True,
        batch_sampler=BatchSamplers.NO_DUPLICATES,
        prompts={"anchor": QUERY_PROMPT},
        logging_steps=10,
        save_strategy="no",
        report_to="wandb",
        run_name=f"vnlegal-lal-ance-round-{round_index}",
    )
    trainer = SentenceTransformerTrainer(
        model=model,
        args=args,
        train_dataset=triplet_dataset,
        loss=train_loss,
    )
    trainer.train()


for round_index in range(1, ANCE_ROUNDS + 1):
    print(f"ANCE round {round_index}/{ANCE_ROUNDS}: encoding corpus and mining hard negatives")
    document_embeddings = encode_corpus(model)
    triplet_dataset = mine_triplets(model, document_embeddings)
    print(f"round {round_index}: mined {len(triplet_dataset)} triplets")
    train_round(round_index, triplet_dataset)
    round_metrics = ir_evaluator(model)
    print(f"round {round_index}: ndcg@10 = {round_metrics.get('alqac_cosine_ndcg@10')}")

/media/caotulab/303A225B3A221DFA/Sang_IIPP/tmp/ipykernel_2612455/3338140651.py:2: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers.losses import CachedMultipleNegativesRankingLoss
/media/caotulab/303A225B3A221DFA/Sang_IIPP/tmp/ipykernel_2612455/3338140651.py:3: DeprecationWarning: Importing from 'sentence_transformers.training_args' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.training_args' instead.
  from sentence_transformers.training_args import BatchSamplers


ANCE round 1/3: encoding corpus and mining hard negatives


Batches:   0%|          | 0/210 [00:00<?, ?it/s]

Batches: 100%|██████████| 3/3 [00:00<00:00, 22.43it/s]
The `warmup_ratio` argument is deprecated in Transformers v5+, and will also be removed from Sentence Transformers once support for Transformers v4 is dropped. Since you're using Transformers v5+, please use `warmup_steps` (as a float) to specify the warmup ratio instead.


round 1: mined 2940 triplets


Step,Training Loss
10,3.985353


KeyboardInterrupt: 

Error in callback <bound method _WandbInit._post_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x7ea3458bdd90>> (for post_run_cell), with arguments args (<ExecutionResult object at 7ea3455c6c50, execution_count=7 error_before_exec=None error_in_exec= info=<ExecutionInfo object at 7ea3458867d0, raw_cell="from sentence_transformers import SentenceTransfor.." transformed_cell="from sentence_transformers import SentenceTransfor.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://ssh-remote%2Bcaotulab_server/home/caotulab/Desktop/Long/ALQAC/notebooks/vnlegal_lal_finetuning.ipynb#X14sdnNjb2RlLXJlbW90ZQ%3D%3D> result=None>,),kwargs {}:


ConnectionResetError: Connection lost

# ***Stage 3: Merge adapter and save***

In [ ]:
model[0].model = model[0].model.merge_and_unload()
model.save_pretrained(MERGED_DIR)

merged_metrics = ir_evaluator(model)
for key in sorted(merged_metrics):
    print(key, merged_metrics[key])

# ***Evaluation test result***

In [ ]:
document_embeddings = model.encode(
    corpus_texts,
    batch_size=16,
    convert_to_tensor=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)

query_ids = list(queries.keys())
query_texts = [queries[qid] for qid in query_ids]
query_embeddings = model.encode(
    query_texts,
    prompt=QUERY_PROMPT,
    batch_size=16,
    convert_to_tensor=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)

similarity = query_embeddings @ document_embeddings.T
ranking = torch.topk(similarity, k=max(K_VALUES), dim=1).indices.cpu().numpy()

table = []
for k in K_VALUES:
    precision_at_k, recall_at_k, accuracy_at_k, f1_at_k = [], [], [], []
    for row, qid in enumerate(query_ids):
        relevant = relevant_docs[qid]
        retrieved = [corpus_ids[index] for index in ranking[row, :k]]
        hits = sum(1 for cid in retrieved if cid in relevant)
        precision = hits / k
        recall = hits / len(relevant) if relevant else 0.0
        accuracy = 1.0 if hits > 0 else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0.0
        precision_at_k.append(precision)
        recall_at_k.append(recall)
        accuracy_at_k.append(accuracy)
        f1_at_k.append(f1)
    table.append(
        (
            k,
            float(np.mean(accuracy_at_k)),
            float(np.mean(precision_at_k)),
            float(np.mean(recall_at_k)),
            float(np.mean(f1_at_k)),
        )
    )

print(f"{'K':>3} | {'Accuracy@K':>10} | {'Precision@K':>11} | {'Recall@K':>9} | {'F1@K':>7}")
for k, accuracy, precision, recall, f1 in table:
    print(f"{k:>3} | {accuracy:>10.4f} | {precision:>11.4f} | {recall:>9.4f} | {f1:>7.4f}")

# ***Push to Hugging Face Hub***

In [ ]:
from huggingface_hub import login

HUB_MODEL_ID = "leonpham1208/alqac_vnlegal_lal"

login()

model.push_to_hub(HUB_MODEL_ID, private=True, exist_ok=True)